# Training base expert on vanilla OGBench environment using BC

In [1]:
import random
import torch
import os
import math

import matplotlib.pyplot as plt

from collections import defaultdict

from causal_gym import PointMazePCH
from causal_rl.algo.imitation.imitate import *
from causal_rl.algo.imitation.finetune import *

<frozen importlib._bootstrap>:241: RuntimeWarning: Your system is avx2 capable but pygame was not built with support for it. The performance of some of your blits could be adversely affected. Consider enabling compile time detection with environment variables like PYGAME_DETECT_AVX2=1 if you are compiling without cross compilation.
/home/et2842/miniconda3/envs/causalenv/lib/python3.11/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


In [2]:
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [16]:
num_steps = 1000
seed = 0
hidden_dims = {'W'}

random.seed(seed)
torch.manual_seed(seed)

In [17]:
env = PointMazePCH(num_steps=num_steps, custom_hidden=hidden_dims, expert_mode=True, seed=seed)
train_eps = env.expert.num_eps
train_eps

1100

In [18]:
X = {f'X{t}' for t in range(num_steps)}
Y = f'Y{num_steps}'
obs_prefix = env.env.observed_unobserved_vars[0]

In [19]:
Z_sets = {}
for Xi in X:
    i = int(Xi[1:])
    cond = set()

    for j in range(i+1):
        cond.update({f'{o}{j}' for o in list(set(obs_prefix) - {'X'})})

    for j in range(i):
        cond.add(f'X{j}')

    Z_sets[Xi] = cond

Z_sets['X1']

{'L0', 'L1', 'P0', 'P1', 'X0'}

In [7]:
records = collect_expert_trajectories(
    env,
    num_episodes=train_eps,
    max_steps=num_steps,
    seed=seed
)

In [20]:
hidden_size = 256
lr = 3e-4
batch_size = 2048
patience = 15
lookback = 1
num_blocks = 4
epochs = 100
dropout = 0.0

dims = {
    'P': 2,
    'L': 2,
    'X': 2
}

In [21]:
model, slots, Z_trim = train_single_policy_long_horizon(
    records,
    Z_sets,
    dims=dims,
    epochs=epochs,
    include_vars=obs_prefix,
    lookback=lookback,
    continuous=True,
    num_actions = env.action_space.shape[0],
    hidden_dim=hidden_size,
    num_blocks=num_blocks,
    dropout=dropout,
    lr=lr,
    batch_size=batch_size,
    patience=patience,
    device=device,
    seed=seed,
    action_bounds=(env.action_space.low, env.action_space.high)
)

policy = shared_policy_fn_long_horizon(model, slots, Z_trim, continuous=True, device=device)
policies = make_shared_policy_dict(policy)

[LongHorizon] Epoch 1: train loss = 0.151249, val loss = 0.134405.
[LongHorizon] Epoch 2: train loss = 0.129798, val loss = 0.124427.
[LongHorizon] Epoch 3: train loss = 0.122895, val loss = 0.120808.
[LongHorizon] Epoch 4: train loss = 0.120221, val loss = 0.119270.
[LongHorizon] Epoch 5: train loss = 0.118432, val loss = 0.117730.
[LongHorizon] Epoch 6: train loss = 0.117156, val loss = 0.116907.
[LongHorizon] Epoch 7: train loss = 0.116219, val loss = 0.115453.
[LongHorizon] Epoch 8: train loss = 0.115173, val loss = 0.114112.
[LongHorizon] Epoch 9: train loss = 0.114534, val loss = 0.114288.
[LongHorizon] Epoch 10: train loss = 0.113681, val loss = 0.113869.
[LongHorizon] Epoch 11: train loss = 0.113181, val loss = 0.112323.
[LongHorizon] Epoch 12: train loss = 0.112540, val loss = 0.112142.
[LongHorizon] Epoch 13: train loss = 0.112033, val loss = 0.111956.
[LongHorizon] Epoch 14: train loss = 0.111512, val loss = 0.110854.
[LongHorizon] Epoch 15: train loss = 0.110952, val loss =

In [22]:
expert_episode_rewards = defaultdict(float)
for rec in records:
    ep = rec['episode']
    expert_episode_rewards[ep] += float(rec['reward'])

num_eps = len(expert_episode_rewards)
expert_rewards = [expert_episode_rewards[e] for e in range(num_eps)]

policy_records = collect_imitator_trajectories(env, policies, num_episodes=num_eps, max_steps=num_steps, seed=seed)
policy_episode_rewards = defaultdict(float)
for rec in policy_records:
    ep = rec['episode']
    policy_episode_rewards[ep] += float(rec['reward'])

policy_rewards = [policy_episode_rewards[e] for e in range(num_eps)]

sum(expert_rewards)/num_eps, sum(policy_rewards)/num_eps

(-937.8136363636364, -9.528599296953793)

In [23]:
# save model for fine-tuning
import os
import torch

SAVE_DIR = '/home/et2842/causal/causalrl/models'
os.makedirs(SAVE_DIR, exist_ok=True)
MODEL_PATH = os.path.join(SAVE_DIR, 'pointmaze_medium_expert.pt')

checkpoint = {
    "state_dict": model.state_dict(),
    "slots": slots,
    "Z_trim": Z_trim,
    "dims": dims,
    "lookback": lookback,
    "continuous": True,
    "num_actions": env.action_space.shape[0],
    "hidden_dim": hidden_size,
    "num_blocks": num_blocks,
    "dropout": 0.0,
    "layernorm": True,
    "final_tanh": True,
    "action_bounds_low": env.action_space.low,
    "action_bounds_high": env.action_space.high,
    "input_dim": int(model.hidden.in_features),
}

torch.save(checkpoint, MODEL_PATH)
print("Saved expert to:", MODEL_PATH)

Saved expert to: /home/et2842/causal/causalrl/models/pointmaze_medium_expert.pt
